# PROCESS SENSOR UNCERTAINTIES

**Requirements**
- Timeseries sensor data
- Information on the types of sensors used for measurement, including uncertainties

This script creates a timeseries of uncertainties for each sensor provided with uncertainty assessment for its measurements.

In [ ]:
import re
import json
import elabapi_python
import pandas         as pd
import numpy          as np
from pathlib          import Path

path_to_sensor_data         = Path.cwd().parent / "data" / "01_Timeseries_Sensors"              / "final"
folder_sensor_data          = Path.cwd().parent / "data" / "01_Timeseries_Sensors"              / "final" / "Operation"
folder_sensor_data_startup  = Path.cwd().parent / "data" / "01_Timeseries_Sensors"              / "final" / "Startup"
folder_sensor_data_shutdown = Path.cwd().parent / "data" / "01_Timeseries_Sensors"              / "final" / "Shutdown"
path_to_uncertainties       = Path.cwd().parent / "data" / "03_Timeseries_Sensor_Uncertainties" / "final"

#### Prepare information query to elab

In [ ]:
#? COMMUNICATION WITH ELABFTW
api_client     = ...
experimentsApi = elabapi_python.ExperimentsApi(api_client)
ItemsApi       = elabapi_python.ItemsApi(api_client)

# list of available experiments and items
experiments              = experimentsApi.read_experiments(limit= 5000)
available_experiment_ids = [exp.id for exp in experiments]
items                    = ItemsApi.read_items(limit= 5000)
available_item_ids       = [item.id for item in items]

In [ ]:
def pt100_tol_class_one_third(T_c):
    """
    Pt100 Class 1/3 DIN (≈ IEC 60751 Class AA):
    ±(0.10 + 0.0017 * |T|) °C
    T_c: scalar, list, or NumPy array (°C).
    """
    T = np.asarray(T_c, dtype=float)
    return 0.10 + 0.0017 * np.abs(T)

def pt100_tol_class_a(T_c):
    """
    IEC 60751, Klasse A:
    ±(0.15 + 0.002 * |T|) °C
    T_c kann Skalar, Liste oder NumPy-Array sein (°C).
    """
    T = np.asarray(T_c, dtype=float)
    return 0.15 + 0.002 * np.abs(T)

def pt100_tol_class_b(T_c):
    """
    IEC 60751, Klasse B:
    ±(0.30 + 0.005 * |T|) °C
    T_c kann Skalar, Liste oder NumPy-Array sein (°C).
    """
    T = np.asarray(T_c, dtype=float)
    return 0.30 + 0.005 * np.abs(T)

## Loop through all sensor data

In [ ]:
for file in folder_sensor_data.glob("*.csv"):
    experiment_id = file.stem
    data_file     = folder_sensor_data / f"{file.stem}.csv"
    # Load the sensor data and retrieve a list of keys
    sensor_data = pd.read_csv(data_file, index_col=0)
    sensor_keys = sensor_data.columns.tolist()

    # Create a Dataframe with the keys as columns and the index as index
    uncertainty_df         = pd.DataFrame(columns=sensor_keys, index=sensor_data.index)

    ########### READ PLANT INFORMATION TO RETRIEVE UNCERTAINTY ASSESSMENTS ###########
    experiment          = experimentsApi.get_experiment(experiment_id)
    experiment_links    = experiment.items_links
    plant_id            = None
    for entry in experiment_links:
        json_entry = entry.__dict__
        if json_entry.get("_category_title") == "Plant Version":
            plant_id    = json_entry.get("_entityid")
            plant_title = json_entry.get("_title")
    if plant_id is None:
        print(f"No plant found for experiment {experiment_id}. Skipping.")
        continue
    else:
        plant_entry     = ItemsApi.get_item(plant_id)

    plant_parts = plant_entry.items_links
    equipment_info = []
    for equipment in plant_parts:
        # If entry is a dict, use it directly; if it's an object, use __dict__
        json_equipment = equipment.__dict__
        if json_equipment.get("_category_title") == "Plant components":
            equipment_id       = json_equipment.get("_entityid")
            equipment_title    = json_equipment.get("_title")
            # Create tuple of id and title
            equipment_info.append((equipment_id, equipment_title))

    for sensor_key in sensor_keys:
        pat = re.compile(rf"\b{re.escape(str(sensor_key))}\b", flags=re.IGNORECASE)
        if any(pat.search(str(info[1])) for info in equipment_info):    
            # Read the id
            equipment_id       = next(info[0] for info in equipment_info if pat.search(str(info[1])))
            equipment_entry    = ItemsApi.get_item(equipment_id)
            equipment_metadata = json.loads(equipment_entry.metadata)
            sensor_type        = equipment_metadata["extra_fields"]["Sensor Type"]["value"]
            sensor_uncertainty = equipment_metadata["extra_fields"]["Uncertainty Estimation"]["value"]

            if sensor_type == "Pt100 class 1/3":
                uncertainty_df[sensor_key] = pt100_tol_class_one_third(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class A":
                uncertainty_df[sensor_key] = pt100_tol_class_a(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class B":
                uncertainty_df[sensor_key] = pt100_tol_class_b(sensor_data[sensor_key])
            else:
                if sensor_uncertainty != "-":
                    try:
                        uncertainty_value          = float(sensor_uncertainty) * 0.01                                                                 # The value is given in percent
                        uncertainty_df[sensor_key] = uncertainty_value * abs(sensor_data[sensor_key])                                                 # If no specific function is defined, use the uncertainty directly
                    except ValueError:
                        print(f"Could not convert uncertainty '{sensor_uncertainty}' for sensor '{sensor_key}' to float. Setting uncertainty to 0.")
                        uncertainty_df[sensor_key] = 0
                else:
                    uncertainty_df[sensor_key] = 0

    uncertainty_df.columns = [f"±Δ{col}" for col in uncertainty_df.columns]                                                                           # Add a "Delta" sign and a \pm sign in front of every column title
    # Save the DataFrame to a CSV file with max 6 decimals
    uncertainty_df.to_csv(path_to_uncertainties / "Operation" / f"{experiment_id}.csv", float_format="%.6f")


In [ ]:
for file in folder_sensor_data_startup.glob("*.csv"):
    experiment_id = file.stem
    # If date "2024_03_22" skip
    if experiment_id in ["134", "151", "239"]:
        print(f"Skipping experiment {experiment_id} as it is not part of the main dataset.")
        continue

    data_file = folder_sensor_data_startup / f"{file.stem}.csv"
    # Load the sensor data and retrieve a list of keys
    sensor_data = pd.read_csv(data_file, index_col=0)
    sensor_keys = sensor_data.columns.tolist()

    # Create a Dataframe with the keys as columns and the index as index
    uncertainty_df         = pd.DataFrame(columns=sensor_keys, index=sensor_data.index)

    ########### READ PLANT INFORMATION TO RETRIEVE UNCERTAINTY ASSESSMENTS ###########
    experiment          = experimentsApi.get_experiment(experiment_id)
    experiment_links    = experiment.items_links
    plant_id            = None
    for entry in experiment_links:
        json_entry = entry.__dict__
        if json_entry.get("_category_title") == "Plant Version":
            plant_id    = json_entry.get("_entityid")
            plant_title = json_entry.get("_title")
    if plant_id is None:
        print(f"No plant found for experiment {experiment.stem}. Skipping.")
        continue
    else:
        plant_entry     = ItemsApi.get_item(plant_id)

    plant_parts = plant_entry.items_links
    equipment_info = []
    for equipment in plant_parts:
        # If entry is a dict, use it directly; if it's an object, use __dict__
        json_equipment = equipment.__dict__
        if json_equipment.get("_category_title") == "Plant components":
            equipment_id       = json_equipment.get("_entityid")
            equipment_title    = json_equipment.get("_title")
            # Create tuple of id and title
            equipment_info.append((equipment_id, equipment_title))

    for sensor_key in sensor_keys:
        pat = re.compile(rf"\b{re.escape(str(sensor_key))}\b", flags=re.IGNORECASE)
        if any(pat.search(str(info[1])) for info in equipment_info):    
            # Read the id
            equipment_id       = next(info[0] for info in equipment_info if pat.search(str(info[1])))
            equipment_entry    = ItemsApi.get_item(equipment_id)
            equipment_metadata = json.loads(equipment_entry.metadata)
            sensor_type        = equipment_metadata["extra_fields"]["Sensor Type"]["value"]
            sensor_uncertainty = equipment_metadata["extra_fields"]["Uncertainty Estimation"]["value"]

            if sensor_type == "Pt100 class 1/3":
                uncertainty_df[sensor_key] = pt100_tol_class_one_third(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class A":
                uncertainty_df[sensor_key] = pt100_tol_class_a(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class B":
                uncertainty_df[sensor_key] = pt100_tol_class_b(sensor_data[sensor_key])
            else:
                if sensor_uncertainty != "-":
                    try:
                        uncertainty_value          = float(sensor_uncertainty) * 0.01                                                                 # The value is given in percent
                        uncertainty_df[sensor_key] = uncertainty_value * abs(sensor_data[sensor_key])                                                 # If no specific function is defined, use the uncertainty directly
                    except ValueError:
                        print(f"Could not convert uncertainty '{sensor_uncertainty}' for sensor '{sensor_key}' to float. Setting uncertainty to 0.")
                        uncertainty_df[sensor_key] = 0
                else:
                    uncertainty_df[sensor_key] = 0

    uncertainty_df.columns = [f"±Δ{col}" for col in uncertainty_df.columns]                                                                           # Add a "Delta" sign and a \pm sign in front of every column title
    # Save the DataFrame to a CSV file with max 6 decimals
    uncertainty_df.to_csv(path_to_uncertainties / "Startup" / f"{experiment_id}.csv", float_format="%.6f")

for file in folder_sensor_data_shutdown.glob("*.csv"):
    experiment_id = file.stem
    data_file     = folder_sensor_data_shutdown / f"{file.stem}.csv"
    # Load the sensor data and retrieve a list of keys
    sensor_data = pd.read_csv(data_file, index_col=0)
    sensor_keys = sensor_data.columns.tolist()

    # Create a Dataframe with the keys as columns and the index as index
    uncertainty_df         = pd.DataFrame(columns=sensor_keys, index=sensor_data.index)

    ########### READ PLANT INFORMATION TO RETRIEVE UNCERTAINTY ASSESSMENTS ###########
    experiment          = experimentsApi.get_experiment(experiment_id)
    experiment_links    = experiment.items_links
    plant_id            = None
    for entry in experiment_links:
        json_entry = entry.__dict__
        if json_entry.get("_category_title") == "Plant Version":
            plant_id    = json_entry.get("_entityid")
            plant_title = json_entry.get("_title")
    if plant_id is None:
        print(f"No plant found for experiment {experiment_id}. Skipping.")
        continue
    else:
        plant_entry     = ItemsApi.get_item(plant_id)

    plant_parts = plant_entry.items_links
    equipment_info = []
    for equipment in plant_parts:
        # If entry is a dict, use it directly; if it's an object, use __dict__
        json_equipment = equipment.__dict__
        if json_equipment.get("_category_title") == "Plant components":
            equipment_id       = json_equipment.get("_entityid")
            equipment_title    = json_equipment.get("_title")
            # Create tuple of id and title
            equipment_info.append((equipment_id, equipment_title))

    for sensor_key in sensor_keys:
        pat = re.compile(rf"\b{re.escape(str(sensor_key))}\b", flags=re.IGNORECASE)
        if any(pat.search(str(info[1])) for info in equipment_info):    
            # Read the id
            equipment_id       = next(info[0] for info in equipment_info if pat.search(str(info[1])))
            equipment_entry    = ItemsApi.get_item(equipment_id)
            equipment_metadata = json.loads(equipment_entry.metadata)
            sensor_type        = equipment_metadata["extra_fields"]["Sensor Type"]["value"]
            sensor_uncertainty = equipment_metadata["extra_fields"]["Uncertainty Estimation"]["value"]

            if sensor_type == "Pt100 class 1/3":
                uncertainty_df[sensor_key] = pt100_tol_class_one_third(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class A":
                uncertainty_df[sensor_key] = pt100_tol_class_a(sensor_data[sensor_key])
            elif sensor_type == "Pt100 class B":
                uncertainty_df[sensor_key] = pt100_tol_class_b(sensor_data[sensor_key])
            else:
                if sensor_uncertainty != "-":
                    try:
                        uncertainty_value          = float(sensor_uncertainty) * 0.01                                                                 # The value is given in percent
                        uncertainty_df[sensor_key] = uncertainty_value * abs(sensor_data[sensor_key])                                                 # If no specific function is defined, use the uncertainty directly
                    except ValueError:
                        print(f"Could not convert uncertainty '{sensor_uncertainty}' for sensor '{sensor_key}' to float. Setting uncertainty to 0.")
                        uncertainty_df[sensor_key] = 0
                else:
                    uncertainty_df[sensor_key] = 0

    uncertainty_df.columns = [f"±Δ{col}" for col in uncertainty_df.columns]                                                                           # Add a "Delta" sign and a \pm sign in front of every column title
    # Save the DataFrame to a CSV file with max 6 decimals
    uncertainty_df.to_csv(path_to_uncertainties / "Shutdown" / f"{experiment_id}.csv", float_format="%.6f")
